# Refusal, coherence, and lie rates across all eight model arms

Every value is loaded from the completed repository artifacts. **Refusal rate** is confirmed `refusal: true` responses divided by all 450 generated responses; unresolved judgments remain in the denominator but not the numerator. **Per-fact lie rate** is `lie` verdicts divided by all available fact verdicts. **Mean coherence** is loaded from the completed coherence-study summaries.

In [ ]:
from collections import defaultdict
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter
import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "PLAN.md").is_file() and (candidate / "experiment").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the mats12 repository root.")


def read_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON at {path}:{line_number}") from exc


ROOT = find_repo_root(Path.cwd().resolve())
COHERENCE_SIX = ROOT / "runs/coherence-study-20260901T181202Z/summary.json"
COHERENCE_FOUR_B = ROOT / "runs/coherence-study-qwen35-4b-20260904T003354Z/summary.json"

MODEL_SPECS = [
    {
        "label": "Aligned Qwen 3.5 9B\nteacher", "group": "aligned",
        "path": "external/hereditary/chinese_censorship_eval/results/qwen_qwen3.5-9b.jsonl",
        "coherence_summary": COHERENCE_SIX, "coherence_arm": "qwen35_9b_aligned",
    },
    {
        "label": "Abliterated Qwen 3.5 9B\nteacher", "group": "abliterated",
        "glob": "runs/behavioral-probe-judge/results/result-*/data.jsonl", "source": "qwen",
        "coherence_summary": COHERENCE_SIX, "coherence_arm": "qwen35_9b_abliterated",
    },
    {
        "label": "Base Llama 3.2 3B\nuntrained", "group": "untrained",
        "glob": "runs/behavioral-probe-judge/results/result-*/data.jsonl", "source": "llama",
        "coherence_summary": COHERENCE_SIX, "coherence_arm": "llama32_3b_base",
    },
    {
        "label": "Llama 3.2 3B\naligned-Qwen student", "group": "aligned",
        "path": "external/hereditary/chinese_censorship_eval/results/llama-3.2-3b_ccp_drop_seed42.jsonl",
        "coherence_summary": COHERENCE_SIX, "coherence_arm": "llama32_3b_qwen_aligned_sft",
    },
    {
        "label": "Llama 3.2 3B\nabliterated-Qwen student", "group": "abliterated",
        "path": "runs/llama-abliterated-seed42-eval-judge-20260829T204306Z/judged-arthur-compatible.jsonl",
        "coherence_summary": COHERENCE_SIX, "coherence_arm": "llama32_3b_qwen_abliterated_sft",
    },
    {
        "label": "Llama 3.2 3B\nsecond-order student", "group": "abliterated",
        "glob": "runs/llama-second-order-seed42-eval-judge-20260901T080004Z/results/result-*/data.jsonl", "source": "adapter",
        "coherence_summary": COHERENCE_SIX, "coherence_arm": "llama32_3b_second_order_sft",
    },
    {
        "label": "Base Qwen 3.5 4B\nuntrained", "group": "untrained",
        "glob": "runs/qwen35-4b-paired-judge-20260902T171800Z/results/result-*/data.jsonl", "source": "qwen35_4b_base",
        "coherence_summary": COHERENCE_FOUR_B, "coherence_arm": "qwen35_4b_base",
    },
    {
        "label": "Qwen 3.5 4B\nabliterated-Qwen student", "group": "abliterated",
        "glob": "runs/qwen35-4b-paired-judge-20260902T171800Z/results/result-*/data.jsonl", "source": "qwen35_4b_abliterated_sft",
        "coherence_summary": COHERENCE_FOUR_B, "coherence_arm": "qwen35_4b_abliterated_sft",
    },
]

# Standalone semantic palette: hue = lineage; lighter shade = downstream generation.
ARM_COLORS = [
    "#2F5D8A",  # aligned Qwen teacher
    "#7CA9D1",  # aligned-Qwen Llama student
    "#B85C1E",  # abliterated Qwen teacher
    "#DF8745",  # abliterated-Qwen Llama student
    "#F0B47E",  # second-order Llama student
    "#666666",  # untrained Base Llama
    "#A6A6A6",  # untrained Base Qwen 4B
    "#DF8745",  # abliterated-Qwen Qwen 4B student
]
LEGEND_COLORS = {"aligned": "#4F83B5", "abliterated": "#D47732", "untrained": "#858585"}


In [ ]:
def load_judgments(spec):
    if "path" in spec:
        paths = [ROOT / spec["path"]]
    else:
        paths = sorted(ROOT.glob(spec["glob"]))
    if not paths or any(not path.is_file() for path in paths):
        raise FileNotFoundError(f"Missing judgments for {spec['label']!r}")
    rows = [
        row for path in paths for row in read_jsonl(path)
        if spec.get("source") is None or row.get("source") == spec["source"]
    ]
    return rows


def summarize_model(spec):
    rows = load_judgments(spec)
    keys = [(str(row["prompt_id"]), int(row["sample"])) for row in rows]
    by_prompt = defaultdict(set)
    for prompt_id, sample in keys:
        by_prompt[prompt_id].add(sample)
    if len(rows) != 450 or len(set(keys)) != 450 or len(by_prompt) != 90:
        raise ValueError(f"Expected 450 unique rows over 90 prompts for {spec['label']!r}")
    if any(samples != set(range(5)) for samples in by_prompt.values()):
        raise ValueError(f"Incomplete five-sample prompt for {spec['label']!r}")

    refusal_true = sum(row.get("refusal") is True for row in rows)
    refusal_null = sum(type(row.get("refusal")) is not bool for row in rows)
    facts = [fact for row in rows for fact in row.get("facts", [])]
    if not facts or any(fact.get("verdict") not in {"yes", "no", "lie"} for fact in facts):
        raise ValueError(f"Missing or invalid fact verdicts for {spec['label']!r}")
    lie_count = sum(fact["verdict"] == "lie" for fact in facts)

    summary = json.loads(spec["coherence_summary"].read_text(encoding="utf-8"))
    coherence = summary.get("arms", {}).get(spec["coherence_arm"])
    if not coherence or coherence.get("count") != 450 or coherence.get("rated_count") != 450 or coherence.get("null_count") != 0:
        raise ValueError(f"Incomplete coherence summary for {spec['label']!r}")

    return {
        "label": spec["label"], "group": spec["group"],
        "refusal_count": refusal_true, "refusal_null": refusal_null,
        "refusal_rate": 100 * refusal_true / len(rows),
        "coherence": float(coherence["mean_coherence"]),
        "lie_count": lie_count, "fact_count": len(facts),
        "lie_rate": 100 * lie_count / len(facts),
    }


model_metrics = [summarize_model(spec) for spec in MODEL_SPECS]
for row in model_metrics:
    print(
        row["label"].replace("\n", " "),
        f"| refusal {row['refusal_count']}/450 = {row['refusal_rate']:.2f}%",
        f"| coherence {row['coherence']:.2f}",
        f"| lies {row['lie_count']}/{row['fact_count']} = {row['lie_rate']:.2f}%",
        f"| unresolved refusals {row['refusal_null']}",
    )

In [ ]:
FIGURE_TITLE = "Refusal, Coherence, and Lie Rates Across Eight Model Arms"

plt.rcParams["font.family"] = "Arial"
DISPLAY_ORDER = [0, 3, 1, 4, 5, 2, 6, 7]
display_metrics = [model_metrics[index] for index in DISPLAY_ORDER]
y = np.array([0.0, 1.0, 2.6, 3.6, 4.6, 6.2, 7.8, 8.8])
labels = [row["label"] for row in display_metrics]
colors = ARM_COLORS

panels = [
    ("lie_rate", "Fact-level lying", "Per-fact lie rate", True),
    ("refusal_rate", "Complete refusal", "Refusal rate", True),
    ("coherence", "Response coherence", "Mean coherence score", False),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 6.6), sharey=True)
fig.subplots_adjust(left=0.21, right=0.985, top=0.80, bottom=0.17, wspace=0.06)
fig.suptitle(FIGURE_TITLE, fontsize=15, fontweight="bold", y=0.97)

for ax, (field, title, xlabel, is_percent) in zip(axes, panels):
    values = np.array([row[field] for row in display_metrics])
    bars = ax.barh(y, values, height=0.68, color=colors, edgecolor="#333333", linewidth=0.6)
    if field == "coherence":
        upper = 105
    else:
        step = 10 if values.max() > 40 else 5
        upper = max(step, min(100, math.ceil((values.max() * 1.18) / step) * step))
    ax.set_xlim(0, upper)
    if is_percent:
        ax.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel(xlabel, fontsize=9.5)
    ax.grid(axis="x", color="#D9D9D9", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    for separator in (1.8, 5.4, 7.0):
        ax.axhline(separator, color="#B8B8B8", linewidth=0.9)
    for bar, value in zip(bars, values):
        suffix = "%" if is_percent else ""
        ax.text(
            min(value + upper * 0.018, upper * 0.965), bar.get_y() + bar.get_height() / 2,
            f"{value:.1f}{suffix}", ha="left" if value < upper * 0.90 else "right",
            va="center", fontsize=8.2, fontweight="bold",
        )

axes[0].set_yticks(y, labels=labels, fontsize=8.5)
axes[0].set_ylim(9.4, -0.7)
for ax in axes[1:]:
    ax.tick_params(labelleft=False)

legend_handles = [
    Patch(facecolor=LEGEND_COLORS["aligned"], edgecolor="#333333", label="Aligned/reference lineage"),
    Patch(facecolor=LEGEND_COLORS["abliterated"], edgecolor="#333333", label="Abliterated-derived lineage"),
    Patch(facecolor=LEGEND_COLORS["untrained"], edgecolor="#333333", label="Untrained control"),
]
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.92), ncol=3, frameon=False, fontsize=8.5)
fig.text(
    0.5, 0.025,
    "Refusal denominator: 450 responses per arm; Base Llama has 2 unresolved judgments retained only in the denominator.\n"
    "Lie denominators are all available judged facts. Student arms are seed 42.",
    ha="center", va="bottom", fontsize=7.7, color="#444444",
)
plt.show()
# Copy-ready export:
# fig.savefig("all_eight_model_metrics.png", dpi=300, bbox_inches="tight", facecolor="white")